# 03 · Preparação para a modelagem

**CRISP-DM fase 3 — preparação dos dados** · Meridiano Imobiliária

---

O notebook [`02-ajustes-dados`](02-ajustes-dados.ipynb) entregou a **base curada**: 2.350
imóveis, 83 colunas, nenhuma célula vazia e cada ajuste com motivo registrado. Ela descreve
a carteira em **unidades de negócio**: pés quadrados, dólares, notas de 0 a 5, e é
justamente isso que a torna legível para a Meridiano e imprópria para um algoritmo baseado
em distância.

In [ ]:
# ============================================================================
# Configuração de avisos
# ============================================================================
import warnings

def warn(*args, **kwargs):
    pass

warnings.warn = warn
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

# ============================================================================
# Importações
# ============================================================================
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
# from category_encoders import CountEncoder

# ============================================================================
# Escalonamento de variáveis numéricas
# ============================================================================
# MinMaxScaler    → ajusta cada variável (coluna) para o intervalo de 0 a 1.
#
# StandardScaler  → centraliza cada variável (coluna) em média 0 e desvio
#                   padrão 1. Não torna a distribuição normal.
#
# Normalizer      → ajusta cada observação (linha) para norma L2 igual a 1,
#                   ou seja, a soma dos quadrados dos valores da linha é 1.

# ============================================================================
# Codificação de variáveis categóricas
# ============================================================================
# OneHotEncoder   → cria uma coluna binária (0/1) para cada categoria.
#                   Indicado para variáveis nominais.
#
# OrdinalEncoder  → converte cada categoria em um número inteiro. A ordem só
#                   é respeitada se informada em `categories`; caso contrário,
#                   segue a ordem alfabética. Indicado para variáveis ordinais.
#
# CountEncoder    → substitui cada categoria pela sua frequência na base
#                   (biblioteca category_encoders).

In [45]:
# Carregando a base de dados, o dicionário e as escalas ordinais
df = pd.read_csv("../data/processed/ames_housing_tratado.csv", sep=';')
dicionario = pd.read_csv("../data/processed/dicionario.csv", sep=';')
escala = pd.read_csv("../data/processed/escala.csv", sep=';', dtype={'nivel': str}).sort_values(['coluna', 'ordem'])
escalas = escala.groupby('coluna')['nivel'].apply(list).to_dict()

In [35]:
# Removendo colunas que não serão utilizadas no modelo
useless_columns = ['ordem', 'pid']

In [36]:
# Seleciona as colunas do dicionário pelo tipo (apenas as presentes na base)
def colunas_do_tipo(*tipos):
    colunas = dicionario.loc[dicionario['Tipo'].isin(tipos), 'variavel']
    return [c for c in colunas if c in df.columns and c not in useless_columns]

# Contínua e discreta
continuous_columns = colunas_do_tipo('Contínua', 'Discreta')

# Ordinal & Nominal
ordinal_columns = colunas_do_tipo('Ordinal')
nominal_columns = colunas_do_tipo('Nominal')

# As derivações (criadas no notebook 02-ajustes-dados)
derived_continuous = ['area_varandas', 'banheiros_total', 'idade_imovel', 'anos_desde_reforma']
derived_binary = ['tem_porao', 'tem_garagem', 'tem_lareira', 'tem_segundo_pavimento', 'foi_reformada']
continuous_columns = list(dict.fromkeys(continuous_columns + derived_continuous + derived_binary))  # sem duplicar

In [37]:
# Ordinais: convertendo os níveis em números na ordem da escala (0, 1, 2, ...)
for col in ordinal_columns:
    niveis = escalas[col]
    if pd.api.types.is_numeric_dtype(df[col]):
        niveis = pd.to_numeric(pd.Series(niveis)).tolist()
    # Rótulos de ausência do notebook 02 ('Sem porão', 'Sem garagem'...) entram como o menor nível
    ausencia = [v for v in df[col].unique() if isinstance(v, str) and v.startswith('Sem ') and v not in niveis]
    niveis = ausencia + niveis
    codigos = pd.Categorical(df[col], categories=niveis, ordered=True).codes
    if (codigos < 0).any():
        raise ValueError(f'{col}: valores fora da escala {set(df.loc[codigos < 0, col])}')
    df[col] = codigos

In [38]:
# Pipelines de pré-processamento para cada tipo de variável
## Contínuas e discretas
continuous_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler())  # Normalização
])

## Ordinais
ordinal_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler())  # Normalização — preserva a ordem e iguala o peso
])

## Nominais
nominal_preprocessor = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),  # Codificação de variáveis
])

# Temporais e colunas não listadas são descartadas (remainder='drop')
preprocessor = ColumnTransformer(transformers=[
    ('continuous', continuous_preprocessor, continuous_columns),
    ('ordinal', ordinal_preprocessor, ordinal_columns),
    ('nominal', nominal_preprocessor, nominal_columns)
], verbose_feature_names_out=False).set_output(transform='pandas')

In [43]:
# Definindo features do processamento
X = df.drop(columns=useless_columns, errors='ignore')

# Aplicando o pré-processamento
X_preprocessed = preprocessor.fit_transform(X)

# Salvando a base pré-processada
X_preprocessed.to_csv("../data/processed/ames_housing_preprocessed.csv", sep=';', index=False)